# syft-ingest API Exploration

In [2]:
from pathlib import Path

import syft_ingest as si

In [5]:
notebook_dir = Path.cwd()
notebook_dir

PosixPath('/Users/khoaguin/Desktop/projects/OpenMined/syfthub-stack/syft-ingest/notebooks')

## 1. Simplified gather() API — YouTube video

In [6]:
# Simplest case: just platform + URLs
corpus = si.gather(
    "youtube", ["https://www.youtube.com/watch?v=zY2dAK-pMPI&t=11s"]
)  # Andrew Trask: talk

print(f"Items fetched: {len(corpus.all_items())}")
if corpus.all_items():
    item = corpus.all_items()[0]
    print(f"  Title: {item.title}")
    print(f"  Author: {item.author}")

2026-04-14 12:07:55.003 | DEBUG    | syft_ingest.sources.youtube:__init__:83 - YtDlpFetcher initialized with config: {'socket_timeout': 30, 'playlistend': 50, 'download_full_video': False}
2026-04-14 12:07:55.003 | INFO     | syft_ingest.core.registry:register_fetcher:79 - Registered fetcher YtDlpFetcher for youtube/yt-dlp
2026-04-14 12:07:55.003 | DEBUG    | syft_ingest.setup:register_fetchers:28 - Registered YtDlpFetcher for YouTube
2026-04-14 12:07:55.003 | INFO     | syft_ingest.core.registry:register_fetcher:79 - Registered fetcher BrightDataFetcher for facebook/brightdata
2026-04-14 12:07:55.004 | INFO     | syft_ingest.core.registry:register_fetcher:79 - Registered fetcher BrightDataFetcher for instagram/brightdata
2026-04-14 12:07:55.004 | DEBUG    | syft_ingest.setup:register_fetchers:37 - Registered BrightDataFetcher for Facebook and Instagram
2026-04-14 12:07:55.004 | INFO     | syft_ingest.core.registry:register_fetcher:79 - Registered fetcher LocalFetcher for local/local
2

Items fetched: 1
  Title: Decentralized AI From Scratch (Part 1: My First P2P AI)
  Author: Andrew Trask


In [14]:
corpus.export(notebook_dir / "data" / "youtube.jsonl")

2026-04-14 12:10:24.600 | INFO     | syft_ingest.core.exporters:_export_jsonl:77 - Exported 1 items to /Users/khoaguin/Desktop/projects/OpenMined/syfthub-stack/syft-ingest/notebooks/data/youtube.jsonl


## 2. Youtube Channel 

In [15]:
# With config options passed as kwargs
corpus = si.gather(
    "youtube",
    ["https://www.youtube.com/@iamtrask"],
    playlistend=3,  # Cap at 3 videos to keep it fast
)

print(f"Videos found: {len(corpus.all_items())}")
for item in corpus.all_items():
    print(f"  [{item.published_at}] {item.title}")

2026-04-14 12:11:38.101 | INFO     | syft_ingest.sources.youtube:fetch:124 - Fetching 1 YouTube URL(s)
2026-04-14 12:11:38.101 | INFO     | syft_ingest.sources.youtube:fetch:135 - Detected channel/playlist URL: https://www.youtube.com/@iamtrask
2026-04-14 12:11:38.101 | DEBUG    | syft_ingest.sources.youtube:_enumerate_channel:314 - Enumerating channel with limit=3
2026-04-14 12:11:38.931 | INFO     | syft_ingest.sources.youtube:_enumerate_channel:351 - Enumerated 3 videos from channel
2026-04-14 12:11:38.931 | INFO     | syft_ingest.sources.youtube:fetch:142 - Enumerated 3 videos from channel
2026-04-14 12:11:38.931 | DEBUG    | syft_ingest.sources.youtube:_extract_video_info_and_captions:404 - Extracting metadata for https://www.youtube.com/watch?v=zY2dAK-pMPI
2026-04-14 12:11:42.322 | DEBUG    | syft_ingest.sources.youtube:_extract_video_info_and_captions:497 - Extracted 882 caption segments for en
2026-04-14 12:11:42.322 | DEBUG    | syft_ingest.sources.youtube:_extract_video_info_

Videos found: 3
  [2026-04-08 00:00:00+00:00] Decentralized AI From Scratch (Part 1: My First P2P AI)
  [2026-03-15 00:00:00+00:00] jobs which are safe from AI: taste, trust, and rare data
  [2018-06-21 00:00:00+00:00] Programming OpenMined.org - Building Federated Learning (4/4)


In [16]:
corpus.export(notebook_dir / "data" / "youtube2.jsonl")

2026-04-14 12:11:49.523 | INFO     | syft_ingest.core.exporters:_export_jsonl:77 - Exported 3 items to /Users/khoaguin/Desktop/projects/OpenMined/syfthub-stack/syft-ingest/notebooks/data/youtube2.jsonl


## 3. Facebook scraping

Requires `BRIGHTDATA_API_TOKEN`. Triggers a real scrape job and polls until done (~60-120s).

In [17]:
import os

token = os.getenv("BRIGHTDATA_API_TOKEN")
print(
    f"BRIGHTDATA_API_TOKEN : {token[:3] + '...' + token[-3:] if token else 'NOT SET -- BrightData cells will fail'}"
)

BRIGHTDATA_API_TOKEN : 7a5...1d5


In [18]:
corpus = await si.async_gather(
    "facebook",
    ["https://www.facebook.com/profile.php?id=61583734012155"],
    author="Syft Influencer Test",
    posts_limit=3,
)

print(f"Items fetched: {len(corpus.all_items())}")
print(f"Author: {corpus.person}")

if corpus.all_items():
    item = corpus.all_items()[0]
    print("\nFirst item:")
    print(f"  Type: {type(item).__name__}")
    print(f"  Title: {item.title}")
    print(f"  Text: {(item.text or '')[:120]}")

2026-04-14 12:12:40.811 | INFO     | syft_ingest.sources.brightdata:fetch_async:113 - Fetching 1 URL(s) for facebook
2026-04-14 12:12:40.812 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:127 - Using platform scraper: facebook
2026-04-14 12:12:41.621 | INFO     | syft_ingest.sources.brightdata:fetch_async:157 - Triggering facebook scrape for https://www.facebook.com/profile.php?id=61583734012155
2026-04-14 12:12:42.007 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:165 - Scrape job created: sd_mnyxvqph27575atm1w
2026-04-14 12:12:42.008 | INFO     | syft_ingest.sources.brightdata:fetch_async:167 - Polling job sd_mnyxvqph27575atm1w with timeout=180s, poll_interval=5s
2026-04-14 12:13:26.055 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:178 - Job sd_mnyxvqph27575atm1w completed
2026-04-14 12:13:27.054 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:182 - Fetched 13340 bytes from job
2026-04-14 12:13:27.055 | INFO     | syft_ingest.sources.brightdata:_pa

Items fetched: 3
Author: Syft Influencer Test

First item:
  Type: ContentItem
  Title: 122107645899124467
  Text: 


In [19]:
corpus.export(notebook_dir / "data" / "fb.jsonl")

2026-04-14 12:13:27.164 | INFO     | syft_ingest.core.exporters:_export_jsonl:77 - Exported 3 items to /Users/khoaguin/Desktop/projects/OpenMined/syfthub-stack/syft-ingest/notebooks/data/fb.jsonl


## 4. Instagram scraping with posts_limit for testing

In [20]:
corpus = await si.async_gather(
    "instagram",
    ["https://www.instagram.com/syft_influencer_test/"],
    author="Syft Influencer Test",
    posts_limit=3,
    timeout=120,
    poll_interval=5,
)

print(f"Items fetched: {len(corpus.all_items())}")
for i, item in enumerate(corpus.all_items()[:3], 1):
    print(f"\n{i}. {type(item).__name__}: {item.title}")

2026-04-14 12:14:06.306 | INFO     | syft_ingest.sources.brightdata:fetch_async:113 - Fetching 1 URL(s) for instagram
2026-04-14 12:14:06.307 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:127 - Using platform scraper: instagram
2026-04-14 12:14:06.559 | INFO     | syft_ingest.sources.brightdata:fetch_async:136 - Searching Instagram posts for https://www.instagram.com/syft_influencer_test/
2026-04-14 12:14:43.053 | DEBUG    | syft_ingest.sources.brightdata:fetch_async:149 - Instagram search completed: sd_mnyxxkea21veisnaer
2026-04-14 12:14:43.054 | INFO     | syft_ingest.sources.brightdata:_parse_response:307 - Limited instagram items to 3 (posts_limit config)
2026-04-14 12:14:43.054 | INFO     | syft_ingest.sources.brightdata:_parse_response:313 - Parsed 3 items from instagram response
2026-04-14 12:14:43.155 | INFO     | syft_ingest.core.gather:async_gather:120 - Gathered 3 items from instagram


Items fetched: 3

1. ContentItem: 3858934126657180181

2. ContentItem: 3858935603245464226

3. ContentItem: 3858933478495298619


In [22]:
corpus.export(notebook_dir / "data" / "ig.jsonl")

2026-04-14 12:14:48.645 | INFO     | syft_ingest.core.exporters:_export_jsonl:77 - Exported 3 items to /Users/khoaguin/Desktop/projects/OpenMined/syfthub-stack/syft-ingest/notebooks/data/ig.jsonl


## 5. Concurrent fetching — async power

Run YouTube, Facebook, and Instagram **simultaneously**. Total time = slowest scrape, not the sum.

In [ ]:
import asyncio
import time

start = time.time()

# All three run concurrently — event loop drives all poll loops simultaneously
corpus_yt, corpus_fb, corpus_ig = await asyncio.gather(
    si.async_gather("youtube", ["https://www.youtube.com/@iamtrask"], playlistend=3),
    si.async_gather(
        "facebook",
        ["https://www.facebook.com/profile.php?id=61583734012155"],
        author="Syft Influencer Test",
        posts_limit=3,
        timeout=300,
    ),
    si.async_gather(
        "instagram",
        ["https://www.instagram.com/syft_influencer_test/"],
        author="Syft Influencer Test",
        posts_limit=3,
        timeout=300,
    ),
)

elapsed = time.time() - start

print(f"YouTube:   {len(corpus_yt.all_items())} items")
print(f"Facebook:  {len(corpus_fb.all_items())} items")
print(f"Instagram: {len(corpus_ig.all_items())} items")
print(f"\nTotal time: {elapsed:.1f}s (sequential would be ~{elapsed * 2:.0f}s+)")

In [ ]:
corpus_yt.export(notebook_dir / "data" / "youtube_async.jsonl")
corpus_fb.export(notebook_dir / "data" / "facebook_async.jsonl")
corpus_ig.export(notebook_dir / "data" / "instagram_async.jsonl")

## 7. Error handling and validation

In [ ]:
# Invalid platform raises ValueError
try:
    corpus = si.gather("tiktok", ["https://tiktok.com/user"])
except KeyError as e:
    print(f"✅ ValueError caught for unsupported platform: {e}")

# Missing URLs raises ValueError
try:
    corpus = si.gather("youtube", None)
except ValueError as e:
    print(f"✅ ValueError caught for missing URLs: {e}")